## Correction notice (2026-08-02)

This notebook originally built the red-noise covariance on a **linearly
spaced** frequency grid. That integral does **not converge** for steep
chromatic spectra: J1909-3744's Band 3 chromatic degradation drifts from
51x at 200 grid points to 18x at 6000, and never settles. The grid below
has been changed to **logarithmic** spacing, which converges by ~400
points and agrees with an independent quadrature rule to 0.5%.

This correction **reversed** the array-level band conclusion, so any
numbers recorded from an earlier run of this notebook are void.

See `scripts/11_grid_convergence.py` and `scripts/12_integrator_crosscheck.py`
for the convergence study. The `scripts/` directory is canonical for all
numbers quoted in the write-up; these notebooks are exploratory.


In [ ]:
# 04_chromatic_noise_band_comparison_J1909.ipynb
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import hasasia.sensitivity as hsen
from hasasia.sensitivity import red_noise_powerlaw, corr_from_psd
from hasasia.utils import create_design_matrix

yr = 365.25*24*3600.0
NU_REF = 1400.0          # MHz — paper's chromatic reference frequency
BAND3, BAND5 = 400.0, 1360.0   # representative band centres (uGMRT)

df = pd.read_csv("inpta_dr2_full_input_table.csv").set_index("psr")

def chromatic_psd(log10A, gamma, chi, freqs, nu_obs, nu_ref=NU_REF):
    """S(f,nu) = A^2/(12pi^2) (f/fyr)^-gamma (nu/nu_ref)^(-2chi) yr^3.
    DM noise: chi=2 (nu^-4). FCN: chi as fitted per pulsar."""
    return red_noise_powerlaw(A=10**log10A, freqs=freqs, gamma=gamma) \
           * (nu_obs/nu_ref)**(-2*chi)

def build_spectrum(row, sigma_us, nu_obs, cad_per_yr=24,
                   use_arn=True, use_dmn=True, use_fcn=True, nfreq=300):
    Tspan = row.span_yr*yr
    N_obs = int(round(row.span_yr*cad_per_yr))
    toas = np.linspace(0, Tspan, N_obs)
    toaerrs = np.ones(N_obs)*sigma_us*1e-6
    rn_freqs = np.logspace(np.log10(1/Tspan), np.log10(cad_per_yr*N_obs/2/Tspan), 800)

    N = np.diag(toaerrs**2)
    if use_arn and np.isfinite(row.arn_log10A):
        N = N + corr_from_psd(rn_freqs,
              red_noise_powerlaw(A=10**row.arn_log10A, freqs=rn_freqs,
                                 gamma=row.arn_gamma), toas)
    if use_dmn and np.isfinite(row.dmn_log10A):
        N = N + corr_from_psd(rn_freqs,
              chromatic_psd(row.dmn_log10A, row.dmn_gamma, 2.0,
                            rn_freqs, nu_obs), toas)
    if use_fcn and np.isfinite(row.fcn_log10A):
        N = N + corr_from_psd(rn_freqs,
              chromatic_psd(row.fcn_log10A, row.fcn_gamma, row.fcn_chi,
                            rn_freqs, nu_obs), toas)

    M = create_design_matrix(toas, RADEC=True, PROPER=True, PX=True)
    psr = hsen.Pulsar(toas, toaerrs, phi=row.phi, theta=row.theta,
                      designmatrix=M, N=N)
    spec = hsen.Spectrum(psr, freqs=np.logspace(np.log10(1/Tspan),
                                                np.log10(1e-7), nfreq))
    _ = spec.NcalInv
    return spec

row = df.loc["J1909-3744"]
b5_wn   = build_spectrum(row, row.toa_b5_us, BAND5, use_arn=False, use_dmn=False, use_fcn=False)
b5_full = build_spectrum(row, row.toa_b5_us, BAND5)
b3_wn   = build_spectrum(row, row.toa_b3_us, BAND3, use_arn=False, use_dmn=False, use_fcn=False)
b3_full = build_spectrum(row, row.toa_b3_us, BAND3)

print(f"Band 5 chromatic degradation: {b5_full.h_c[0]/b5_wn.h_c[0]:.2f}x")
print(f"Band 3 chromatic degradation: {b3_full.h_c[0]/b3_wn.h_c[0]:.2f}x")
print(f"Net low-f Band3/Band5: {b3_full.h_c[0]/b5_full.h_c[0]:.2f} (>1 => Band 5 better)")

plt.figure(figsize=(7.5,5.5))
plt.loglog(b3_wn.freqs, b3_wn.h_c, ':',  c='C0', label='Band 3, white only')
plt.loglog(b3_full.freqs, b3_full.h_c, '-', c='C0', label='Band 3, full noise')
plt.loglog(b5_wn.freqs, b5_wn.h_c, ':',  c='C1', label='Band 5, white only')
plt.loglog(b5_full.freqs, b5_full.h_c, '-', c='C1', label='Band 5, full noise')
plt.xlabel('Frequency [Hz]'); plt.ylabel(r'Characteristic strain $h_c$')
plt.title(r'J1909$-$3744: chromatic noise penalty by uGMRT band')
plt.legend(fontsize=9); plt.grid(True, which='both', alpha=0.3)
plt.savefig("J1909_band_comparison.png", dpi=150, bbox_inches='tight')
plt.show()